In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/content/dresses.csv')

In [3]:
df.head()      # shows first 5 rows
df.tail()      # shows last 5 rows
df.shape       # tells you (515, 12) → 515 rows, 12 columns
df.dtypes      # tells you what type each column is (number? text? date?)
df.describe()  # gives you min, max, average, etc. for every column

,product_id,rating,ratings_count,initial_price,discount
count,1.000000e+02,100.000000,100.000000,100.000000,92.000000
mean,1.787220e+07,3.527000,154.620000,2557.300000,58.836957
std,3.927623e+06,1.388688,596.876546,1231.282295,14.313044
min,1.341220e+06,0.000000,0.000000,699.000000,10.000000
25%,1.363408e+07,3.575000,6.000000,1799.000000,52.000000
50%,1.896292e+07,4.000000,12.000000,2249.500000,63.000000
75%,2.162574e+07,4.300000,43.500000,2999.000000,68.250000
max,2.263854e+07,5.000000,4441.000000,7828.000000,83.000000


In [5]:
print(df.columns.tolist())
print(df.dtypes)

['product_id', 'title', 'product_description', 'rating', 'ratings_count', 'initial_price', 'discount', 'final_price', 'currency', 'images', 'delivery_options', 'product_details', 'breadcrumbs', 'product_specifications', 'amount_of_stars', 'what_customers_said', 'seller_name', 'sizes', 'videos', 'seller_information', 'variations', 'best_offer', 'more_offers', 'category']
product_id                  int64
title                      object
product_description        object
rating                    float64
ratings_count               int64
initial_price             float64
discount                  float64
final_price                object
currency                   object
images                     object
delivery_options           object
product_details            object
breadcrumbs                object
product_specifications     object
amount_of_stars            object
what_customers_said        object
seller_name                object
sizes                      object
videos         

In [4]:
df.isnull().sum()

,0
product_id,0
title,0
product_description,0
rating,0
ratings_count,0
initial_price,0
discount,8
final_price,0
currency,0
images,0


In [14]:
print(f"Shape BEFORE: {df.shape}")

# discount is a number — 0 means no discount
df['discount'] = df['discount'].fillna(0)

# text columns — fill with 'Not Available'
df['what_customers_said'] = df['what_customers_said'].fillna('Not Available')
df['seller_name']         = df['seller_name'].fillna('Not Available')
df['seller_information']  = df['seller_information'].fillna('Not Available')
df['variations']          = df['variations'].fillna('Not Available')
df['videos']              = df['videos'].fillna('No Video')

print(f"Shape AFTER:  {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

Shape BEFORE: (100, 25)
Shape AFTER:  (100, 25)
Remaining missing values: 0


In [15]:
df['final_price'] = (df['final_price']
                     .astype(str)
                     .str.replace('"', '', regex=False)
                     .str.replace('₹', '', regex=False)
                     .str.replace(',', '', regex=False)
                     .str.strip()
                     .astype(float))

print(f"final_price cleaned!")
print(f"final_price dtype: {df['final_price'].dtype}")
print(df[['initial_price', 'final_price']].head(5))

final_price cleaned!
final_price dtype: float64
   initial_price  final_price
0         2999.0       2999.0
1         4399.0       2419.0
2         3490.0       2094.0
3         3599.0       2519.0
4         1299.0        628.0


In [16]:
# Filter 1: High rated products
high_rated = df[df['rating'] >= 4]
print(f"High-rated products (rating ≥ 4): {len(high_rated)} rows")

# Filter 2: Products that have a discount
discounted = df[df['discount'] > 0]
print(f"Products with discount: {len(discounted)} rows")

# Filter 3: Popular products (more than 1000 ratings)
popular = df[df['ratings_count'] > 1000]
print(f"Popular products (>1000 ratings): {len(popular)} rows")

High-rated products (rating ≥ 4): 56 rows
Products with discount: 92 rows
Popular products (>1000 ratings): 3 rows


In [17]:
selected = df[['title', 'category', 'initial_price',
               'final_price', 'discount', 'rating',
               'ratings_count', 'seller_name']]

print("Selected columns:")
selected.head(8)

Selected columns:


,title,category,initial_price,final_price,discount,rating,ratings_count,seller_name
0,Curvy Clan,dresses,2999.0,2999.0,0.0,3.7,3,RUB███S C███
1,Trendyol,dresses,4399.0,2419.0,45.0,4.1,10,Sup███om ███
2,COVER STORY,dresses,3490.0,2094.0,40.0,4.8,24,Cov███Sto███Clo█████████ite███
3,Trendyol,dresses,3599.0,2519.0,30.0,3.9,35,Sup███om ███
4,RACHNA,dresses,1299.0,628.0,51.0,3.0,4,RAC███ AR███RIN█████████TD.███
5,Label Ritu Kumar,dresses,5900.0,4130.0,30.0,0.0,0,REL███CE ███U K█████████ATE█████████
6,Harpa,dresses,1999.0,899.0,55.0,2.3,4,HAR███RET███ PV█████████
7,aarke Ritu Kumar,dresses,2200.0,1540.0,30.0,4.0,6,REL███CE ███U K█████████ATE█████████


In [18]:
print(f"Shape BEFORE: {df.shape}")
dup_count = df.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")

df = df.drop_duplicates().reset_index(drop=True)

print(f"Shape AFTER:  {df.shape}")
print(f"{dup_count} duplicate rows removed!")

Shape BEFORE: (100, 25)
Duplicate rows found: 0
Shape AFTER:  (100, 25)
0 duplicate rows removed!


In [19]:
# total_amount = final_price × ratings_count
# (ratings_count represents how many people bought/reviewed it)
df['total_amount'] = (df['final_price'] * df['ratings_count']).round(2)

print("Derived column 'total_amount' created!")
print(f"\ntotal_amount Summary:")
print(f"   Min   : ₹{df['total_amount'].min():,.2f}")
print(f"   Max   : ₹{df['total_amount'].max():,.2f}")
print(f"   Mean  : ₹{df['total_amount'].mean():,.2f}")
print()
df[['title', 'initial_price', 'final_price',
    'ratings_count', 'total_amount']].head(8)

Derived column 'total_amount' created!

total_amount Summary:
   Min   : ₹0.00
   Max   : ₹4,248,843.00
   Mean  : ₹140,342.60



,title,initial_price,final_price,ratings_count,total_amount
0,Curvy Clan,2999.0,2999.0,3,8997.0
1,Trendyol,4399.0,2419.0,10,24190.0
2,COVER STORY,3490.0,2094.0,24,50256.0
3,Trendyol,3599.0,2519.0,35,88165.0
4,RACHNA,1299.0,628.0,4,2512.0
5,Label Ritu Kumar,5900.0,4130.0,0,0.0
6,Harpa,1999.0,899.0,4,3596.0
7,aarke Ritu Kumar,2200.0,1540.0,6,9240.0


In [20]:
output_path = 'shopping_cleaned.csv'
df.to_csv(output_path, index=False)

print(f" Cleaned dataset saved!")
print(f"   Rows       : {df.shape[0]}")
print(f"   Columns    : {df.shape[1]}")
print(f"   Missing    : {df.isnull().sum().sum()}")
print(f"   Duplicates : {df.duplicated().sum()}")

 Cleaned dataset saved!
   Rows       : 100
   Columns    : 25
   Missing    : 0
   Duplicates : 0


In [21]:
# Download the file to your computer
from google.colab import files
files.download('shopping_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>